In [28]:
!pip install kaggle

Dependency Imports

os – Used to interact with the operating system, such as managing file paths and directories.

json – Helps read and write JSON formatted data used for configurations or metadata.

ZipFile – Used to extract and handle datasets that are stored in ZIP files.

pandas – Provides data structures (DataFrames) for loading, inspecting, and processing datasets.

train_test_split – Splits the dataset into training and testing sets to evaluate model performance.

Sequential – Used to build the neural network model layer by layer.

Dense – Fully connected layer used for final classification of sentiment.

Embedding – Converts words into numerical vector representations for the neural network.

LSTM – A recurrent neural network layer that captures sequential context in text data.

Tokenizer – Converts text sentences into numerical tokens that can be used by the model.

pad_sequences – Ensures all input sequences have the same length by adding padding where necessary.

In [29]:
import os
import json

from zipfile import ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding,LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


Data Collection & Load Kaggle API Credentials

The kaggle.json file contains the Kaggle API credentials required to access datasets from Kaggle.

json.load() reads the JSON file and converts it into a Python dictionary.

The credentials stored in this dictionary are later used to authenticate and download datasets programmatically from Kaggle.

In [30]:
kaggle_dictionary = json.load(open("kaggle.json"))

Set up kaggle credentials as environment variables

In [31]:
kaggle_dictionary.keys()

dict_keys(['username', 'key'])

In [32]:
os.environ["KAGGLE_USERNAME"] = kaggle_dictionary["username"]
os.environ["KAGGLE_KEY"] = kaggle_dictionary["key"]

In [33]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
imdb-dataset-of-50k-movie-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)


Extract the Downloaded Dataset

The dataset downloaded from Kaggle is stored as a ZIP file.

ZipFile() is used to open the compressed file in read mode ("r").

extractall() extracts all files from the ZIP archive into the current working directory.

This step makes the dataset files accessible for loading and preprocessing in the next steps.

In [34]:
with ZipFile("imdb-dataset-of-50k-movie-reviews.zip", "r") as zip_ref:
  zip_ref.extractall()

Loading the Dataset

In [35]:
data = pd.read_csv("/content/IMDB Dataset.csv")

In [36]:
data.shape

(50000, 2)

In [37]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [38]:
data.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [39]:
data['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


Convert Sentiment Labels to Numeric Values

The dataset contains sentiment labels as text values (positive, negative).

Machine learning models require numerical inputs, so these labels are converted to numbers.

positive is mapped to 1 and negative is mapped to 0.

inplace=True updates the dataset directly without creating a new copy.

This step prepares the target variable so the model can learn and predict sentiment correctly.

In [40]:
data.replace({"sentiment": {"positive": 1,"negative": 0}}, inplace=True)

/tmp/ipykernel_17746/3591717516.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({"sentiment": {"positive": 1,"negative": 0}}, inplace=True)


In [41]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [42]:
data['sentiment'].value_counts()

,count
sentiment,
1,25000
0,25000


Split the dataset into training and test data

In [43]:
train_data,test_data = train_test_split(data,test_size = 0.2,random_state = 42)

In [44]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


Data Preprocessing

Text Tokenization and Sequence Preparation

Tokenizer converts text reviews into numerical tokens so they can be processed by the neural network.

num_words=5000 limits the vocabulary to the 5000 most frequent words, which helps reduce complexity and training time.

Why add oov_token
If a new word appears during prediction, it maps to <OOV> instead of being ignored.

fit_on_texts() builds a word index by learning the vocabulary from the training reviews.

texts_to_sequences() converts each review into a sequence of integers representing words.

pad_sequences() ensures all sequences have the same length (200 words) by adding padding where necessary.

This step transforms raw text reviews into fixed-length numerical sequences suitable for input to the LSTM model.

In [45]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_data["review"])

X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [46]:
print(X_train)

[[ 145 1084   17 ...  206  352 3857]
 [ 311    6  426 ...   90  104   10]
 [   0    0    0 ...    3  711   63]
 ...
 [   0    0    0 ... 1642    3  604]
 [   0    0    0 ...  126    1    1]
 [   0    0    0 ...   71   74 2063]]


In [47]:
print(X_test)

[[   1  697    6 ...  996  720  156]
 [  45  201 1568 ...    8    1    1]
 [   0    0    0 ...   51 1089   97]
 ...
 [  20 2069  204 ...  126  201 3242]
 [   0    0    0 ... 1067    2 2306]
 [   0    0    0 ...    2  333   28]]


In [48]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [49]:
print(Y_test)

33553    1
9427     1
199      0
12447    1
39489    0
        ..
28567    0
25079    1
18707    1
15200    0
5857     1
Name: sentiment, Length: 10000, dtype: int64


In [50]:
print(Y_train)

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


LSTM- Long Short-Term Memory


---
Model Architecture

A Sequential model is used to build the neural network layer-by-layer for sentiment classification.

The Embedding layer converts each word index into a dense vector representation so the model can learn relationships between words.

input_dim=5000 represents the vocabulary size used by the tokenizer.

output_dim=128 means each word is represented by a vector of 128 features.

input_length=200 defines the fixed length of each review after padding.

The LSTM layer processes the sequence of word embeddings and captures contextual dependencies between words in the review.

128 units allow the model to learn complex patterns in the text.

dropout=0.2 and recurrent_dropout=0.2 help prevent overfitting by randomly disabling some connections during training.

The Dense output layer performs the final classification.

A single neuron is used because the problem is binary sentiment classification (positive or negative).

The sigmoid activation function outputs a probability between 0 and 1 representing the predicted sentiment.





In [51]:
## build the model

model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128,input_length=200))
model.add(LSTM(128,dropout=0.2,recurrent_dropout=0.2))
model.add(Dense(1,activation="sigmoid"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [52]:
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [53]:
model.build(input_shape=(None, 200))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

Training the model

In [54]:
model.fit(X_train,Y_train,epochs=5, batch_size = 64,validation_split = 0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 214s 422ms/step - accuracy: 0.7690 - loss: 0.4772 - val_accuracy: 0.8545 - val_loss: 0.3499
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 209s 418ms/step - accuracy: 0.8471 - loss: 0.3587 - val_accuracy: 0.8171 - val_loss: 0.4650
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 208s 416ms/step - accuracy: 0.8455 - loss: 0.3601 - val_accuracy: 0.8540 - val_loss: 0.3572
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 208s 416ms/step - accuracy: 0.8807 - loss: 0.2960 - val_accuracy: 0.8708 - val_loss: 0.3263
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 207s 414ms/step - accuracy: 0.9006 - loss: 0.2508 - val_accuracy: 0.8751 - val_loss: 0.3125


Model evalution

In [55]:
loss,accuracy = model.evaluate(X_test,Y_test)

print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 120ms/step - accuracy: 0.8807 - loss: 0.2979
Test Loss: 0.29789403080940247
Test Accuracy: 0.8806999921798706


Building a predictive system

In [56]:
def predict_sentiment(review):
  ## tokenize and pad the review
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence,maxlen = 200)
  prediction = model.predict(padded_sequence)
  sentiment = "positive" if prediction > 0.5 else "negative"
  return sentiment

In [57]:
## example usage
new_review = "This movie was fantastic. I loved it."
sentiment = predict_sentiment(new_review)
print(f"The sentiment the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step
The sentiment the review is: positive


In [58]:
## example usage
new_review = "This movie was not good"
sentiment = predict_sentiment(new_review)
print(f"The sentiment the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
The sentiment the review is: negative


In [59]:
import pickle

# save model
model.save("sentiment_model.h5")

# save tokenizer
with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

In [60]:
from google.colab import files

files.download("sentiment_model.h5")
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>